In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az 

# define random seed for reproducibility
random_seed = 42

In [3]:
df = pd.read_csv('data/scr_amg_hipp_all_PH.csv')
df_group = pd.read_csv('data/demographic.csv')
df_group.head()

,sub_id,group,Gender,Age
0,sub-010,VCC,2.0,29.0
1,sub-016,VCC,1.0,25.0
2,sub-021,VPTSD,1.0,40.0
3,sub-026,VPTSD,1.0,32.0
4,sub-027,VCC,1.0,35.0


In [4]:
df = pd.merge(df, df_group[['sub_id','group']], left_on='sub', right_on='sub_id')
df.head()

,sub,Condition,Event.Nr,CDA.AmpSum,pe,scr,index,subject,trialNo,condition,coupling,amg,amg_vmpfc,sub_id,group
0,sub-189,CSplusUS1,1,0.2852,0.500000,0.2852,3036,sub-189,1,CSplusUS1,0.904762,0.476625,0.833333,sub-189,HC
1,sub-189,CSminus1,2,0.1033,-0.500000,0.1033,3037,sub-189,2,CSminus1,-0.380952,0.081692,0.428571,sub-189,HC
2,sub-189,CSplus1,3,0.0783,-0.750000,0.0783,3038,sub-189,3,CSplus1,0.571429,-0.219659,0.690476,sub-189,HC
3,sub-189,CSplusUS1,4,0.1772,0.686106,0.1772,3039,sub-189,4,CSplusUS1,0.619048,0.006618,0.880952,sub-189,HC
4,sub-189,CSminus1,5,0.0000,-0.250000,0.0000,3040,sub-189,5,CSminus1,0.833333,-0.188212,0.595238,sub-189,HC


In [5]:
# read file
# only for no shock
# %% amygdala-hippocampus coupling pymc model
# Encode 'sub' as integer indices
df['sub_idx'] = pd.Categorical(df['sub']).codes
n_subs = df['sub_idx'].nunique()

# Encode 'group' as integer indices (make ordering explicit!)
# Data uses: HC (healthy controls), VCC (combat controls), VPTSD (PTSD)
group_order = ['HC', 'VCC', 'VPTSD']
df['group'] = pd.Categorical(df['group'], categories=group_order, ordered=True)
df['group_idx'] = df['group'].cat.codes
n_groups = df['group_idx'].nunique()

# Check which group is reference (index 0)
print("Group coding (0 = reference):", {g: i for i, g in enumerate(df['group'].cat.categories)})

# Extract variables
pe = df['pe'].values
coupling = df['coupling'].values
amg = df['amg'].values
trialNo = df['trialNo'].values
sub_idx = df['sub_idx'].values
group_idx = df['group_idx'].values

Group coding (0 = reference): {'HC': 0, 'VCC': 1, 'VPTSD': 2}


In [6]:
with pm.Model() as model:
    
    # Fixed effects (main effects)
    beta_coupling = pm.Normal('beta_coupling', mu=0, sigma=1)  # Coupling effect for reference group
    beta_amg = pm.Normal('beta_amg', mu=0, sigma=1)
    beta_trialNo = pm.Normal('beta_trialNo', mu=0, sigma=1)
    
    # Main effect of group (reference group = 0, so n_groups-1 parameters)
    beta_group_raw = pm.Normal('beta_group_raw', mu=0, sigma=1, shape=n_groups - 1)
    beta_group = pm.math.concatenate([[0], beta_group_raw])  # Pad reference group with 0
    
    # Group × Coupling interaction (deviation from reference group's slope)
    beta_interaction_raw = pm.Normal('beta_interaction_raw', mu=0, sigma=1, shape=n_groups - 1)
    beta_interaction = pm.math.concatenate([[0], beta_interaction_raw])  # Pad reference with 0
    
    # Hyperpriors for random intercepts
    mu_a = pm.Normal('mu_a', mu=0, sigma=1)
    sigma_a = pm.HalfNormal('sigma_a', sigma=1)
    
    # Non-centered random intercepts
    z_a = pm.Normal('z_a', mu=0, sigma=1, shape=n_subs)
    a = pm.Deterministic('a', mu_a + z_a * sigma_a)
    
    # Expected value of outcome
    mu = (
        a[sub_idx] +
        beta_group[group_idx] +                        # Main effect of group
        beta_coupling * coupling +                      # Main effect of coupling (reference slope)
        beta_interaction[group_idx] * coupling +        # Interaction: group-specific slope adjustment
        beta_amg * amg +
        beta_trialNo * trialNo
    )
    
    # Likelihood
    sigma = pm.HalfNormal('sigma', sigma=1)
    y_obs = pm.Normal('pe', mu=mu, sigma=sigma, observed=pe)
    
    trace = pm.sample(chains=4, random_seed=random_seed, return_inferencedata=True,
                      idata_kwargs={"log_likelihood": True})

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta_coupling, beta_amg, beta_trialNo, beta_group_raw, beta_interaction_raw, mu_a, sigma_a, z_a, sigma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 11 seconds.


In [7]:
sum_hipp = az.summary(trace, var_names=['beta_coupling', 'beta_group_raw', 'beta_interaction_raw'], hdi_prob=0.89)
print(f'Summary of Amygdala hippocampus coupling model: {sum_hipp}')

# %% grab group specific slopes for coupling
# group specific slope

# Post-hoc calculation of group-specific slopes
posterior = trace.posterior
slope_hipp_HC = posterior['beta_coupling']
if n_groups != 3:
    raise ValueError(f"Expected 3 groups {group_order}, but found n_groups={n_groups}.")
slope_hipp_VCC = posterior['beta_coupling'] + posterior['beta_interaction_raw'][:, :, group_order.index('VCC') - 1]
slope_hipp_VPTSD = posterior['beta_coupling'] + posterior['beta_interaction_raw'][:, :, group_order.index('VPTSD') - 1]

print("HC slope (mean, SD, HDI):", float(slope_hipp_HC.mean()), float(slope_hipp_HC.std()), az.hdi(slope_hipp_HC.values.flatten(), hdi_prob=0.89))
print("VCC slope (mean, SD, HDI):", float(slope_hipp_VCC.mean()), float(slope_hipp_VCC.std()), az.hdi(slope_hipp_VCC.values.flatten(), hdi_prob=0.89))
print("VPTSD slope (mean, SD, HDI):", float(slope_hipp_VPTSD.mean()), float(slope_hipp_VPTSD.std()), az.hdi(slope_hipp_VPTSD.values.flatten(), hdi_prob=0.89))

Summary of Amygdala hippocampus coupling model:                           mean     sd  hdi_5.5%  hdi_94.5%  mcse_mean  \
beta_coupling            0.129  0.039     0.067      0.192      0.001   
beta_group_raw[0]        0.011  0.035    -0.043      0.067      0.001   
beta_group_raw[1]        0.018  0.033    -0.035      0.071      0.001   
beta_interaction_raw[0] -0.036  0.053    -0.120      0.048      0.001   
beta_interaction_raw[1] -0.041  0.053    -0.124      0.044      0.001   

                         mcse_sd  ess_bulk  ess_tail  r_hat  
beta_coupling              0.001    1523.0    2327.0    1.0  
beta_group_raw[0]          0.001    1882.0    2382.0    1.0  
beta_group_raw[1]          0.001    1740.0    2569.0    1.0  
beta_interaction_raw[0]    0.001    1805.0    2418.0    1.0  
beta_interaction_raw[1]    0.001    1794.0    2417.0    1.0  
HC slope (mean, SD, HDI): 0.12913204692019045 0.03899155621036434 [0.0672673  0.19185388]
VCC slope (mean, SD, HDI): 0.09328184776155149 0.03

In [8]:
np.mean(slope_hipp_VCC > 0)

<xarray.DataArray ()> Size: 8B
array(0.99675)
Coordinates:
    beta_interaction_raw_dim_0  int64 8B 0

# Amygdala-vmPFC

In [9]:
df['sub_idx'] = pd.Categorical(df['sub']).codes
n_subs = df['sub_idx'].nunique()

# Encode 'group' as integer indices (keep the same explicit ordering)
df['group'] = pd.Categorical(df['group'], categories=group_order, ordered=True)
df['group_idx'] = df['group'].cat.codes
n_groups = df['group_idx'].nunique()

# Check which group is reference (index 0)
print("Group coding (0 = reference):", {g: i for i, g in enumerate(df['group'].cat.categories)})
# %%
coupling = df['amg_vmpfc'].values
amg = df['amg'].values
trialNo = df['trialNo'].values
sub_idx = df['sub_idx'].values
group_idx = df['group_idx'].values

with pm.Model() as model_vmpfc:
    
    # Fixed effects (main effects)
    beta_coupling = pm.Normal('beta_coupling', mu=0, sigma=1)  # Coupling effect for reference group
    beta_amg = pm.Normal('beta_amg', mu=0, sigma=1)
    beta_trialNo = pm.Normal('beta_trialNo', mu=0, sigma=1)
    
    # Main effect of group (reference group = 0, so n_groups-1 parameters)
    beta_group_raw = pm.Normal('beta_group_raw', mu=0, sigma=1, shape=n_groups - 1)
    beta_group = pm.math.concatenate([[0], beta_group_raw])  # Pad reference group with 0
    
    # Group × Coupling interaction (deviation from reference group's slope)
    beta_interaction_raw = pm.Normal('beta_interaction_raw', mu=0, sigma=1, shape=n_groups - 1)
    beta_interaction = pm.math.concatenate([[0], beta_interaction_raw])  # Pad reference with 0
    
    # Hyperpriors for random intercepts
    mu_a = pm.Normal('mu_a', mu=0, sigma=1)
    sigma_a = pm.HalfNormal('sigma_a', sigma=1)
    
    # Non-centered random intercepts
    z_a = pm.Normal('z_a', mu=0, sigma=1, shape=n_subs)
    a = pm.Deterministic('a', mu_a + z_a * sigma_a)
    
    # Expected value of outcome
    mu = (
        a[sub_idx] +
        beta_group[group_idx] +                        # Main effect of group
        beta_coupling * coupling +                      # Main effect of coupling (reference slope)
        beta_interaction[group_idx] * coupling +        # Interaction: group-specific slope adjustment
        beta_amg * amg +
        beta_trialNo * trialNo
    )
    
    # Likelihood
    sigma = pm.HalfNormal('sigma', sigma=1)
    y_obs = pm.Normal('pe', mu=mu, sigma=sigma, observed=pe)
    
    trace_vmpfc = pm.sample(chains=4, random_seed=random_seed, return_inferencedata=True,
                      idata_kwargs={"log_likelihood": True})


Group coding (0 = reference): {'HC': 0, 'VCC': 1, 'VPTSD': 2}


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta_coupling, beta_amg, beta_trialNo, beta_group_raw, beta_interaction_raw, mu_a, sigma_a, z_a, sigma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 9 seconds.


In [10]:
sum_vmpfc = az.summary(trace_vmpfc, var_names=['beta_coupling', 'beta_group_raw', 'beta_interaction_raw'], hdi_prob=0.89)
print(f"Summary of Amygdala-vmPFC coupling model: {sum_vmpfc}")
# %% grab group specific slopes for coupling
# group specific slope

# Post-hoc calculation of group-specific slopes
posterior = trace_vmpfc.posterior
slope_vmpfc_HC = posterior['beta_coupling']
if n_groups != 3:
    raise ValueError(f"Expected 3 groups {group_order}, but found n_groups={n_groups}.")
slope_vmpfc_VCC = posterior['beta_coupling'] + posterior['beta_interaction_raw'][:, :, group_order.index('VCC') - 1]
slope_vmpfc_VPTSD = posterior['beta_coupling'] + posterior['beta_interaction_raw'][:, :, group_order.index('VPTSD') - 1]

print("HC slope mean (SD, HDI):", float(slope_vmpfc_HC.mean()), float(slope_vmpfc_HC.std()), az.hdi(slope_vmpfc_HC.values.flatten(), hdi_prob=0.89))
print("VCC slope mean (SD, HDI):", float(slope_vmpfc_VCC.mean()), float(slope_vmpfc_VCC.std()), az.hdi(slope_vmpfc_VCC.values.flatten(), hdi_prob=0.89))
print("VPTSD slope mean (SD, HDI):", float(slope_vmpfc_VPTSD.mean()), float(slope_vmpfc_VPTSD.std()), az.hdi(slope_vmpfc_VPTSD.values.flatten(), hdi_prob=0.89))

Summary of Amygdala-vmPFC coupling model:                           mean     sd  hdi_5.5%  hdi_94.5%  mcse_mean  \
beta_coupling           -0.044  0.032    -0.092      0.012      0.001   
beta_group_raw[0]        0.012  0.023    -0.023      0.049      0.000   
beta_group_raw[1]       -0.034  0.022    -0.068      0.003      0.000   
beta_interaction_raw[0] -0.038  0.043    -0.101      0.037      0.001   
beta_interaction_raw[1]  0.079  0.044     0.006      0.145      0.001   

                         mcse_sd  ess_bulk  ess_tail  r_hat  
beta_coupling              0.000    2311.0    2914.0    1.0  
beta_group_raw[0]          0.000    2833.0    3038.0    1.0  
beta_group_raw[1]          0.000    3136.0    3572.0    1.0  
beta_interaction_raw[0]    0.001    2639.0    3071.0    1.0  
beta_interaction_raw[1]    0.001    3164.0    3283.0    1.0  
HC slope mean (SD, HDI): -0.043764359476404016 0.03223372377192254 [-0.09195268  0.01175622]
VCC slope mean (SD, HDI): -0.08148479685400954 0.02863

In [11]:
# Amygdala-vmPFC: combat-exposed without PTSD vs. with PTSD
diff_vmpfc = slope_vmpfc_VCC - slope_vmpfc_VPTSD
pd_diff = float((diff_vmpfc < 0).mean())  # VCC more negative than PTSD
hdi_diff = az.hdi(diff_vmpfc.values.flatten(), hdi_prob=0.89)
print(f"VCC - VPTSD vmpfc: mean={float(diff_vmpfc.mean()):.3f}, sd = {float(diff_vmpfc.std()):.3f}, pd={pd_diff*100:.1f}%, 89% HDI={hdi_diff}")


VCC - VPTSD vmpfc: mean=-0.116, sd = 0.041, pd=99.6%, 89% HDI=[-0.18344913 -0.0523584 ]


In [12]:

# Amygdala-hippocampus: combat-exposed without PTSD vs. with PTSD
diff_hipp = slope_hipp_VCC - slope_hipp_VPTSD
pd_diff = float((diff_hipp > 0).mean())  # VCC more negative than PTSD
hdi_diff = az.hdi(diff_hipp.values.flatten(), hdi_prob=0.89)
print(f"VCC - VPTSD hipp: mean={float(diff_hipp.mean()):.3f}, sd = {float(diff_hipp.std()):.3f}, pd={pd_diff*100:.1f}%, 89% HDI={hdi_diff}")


VCC - VPTSD hipp: mean=0.006, sd = 0.052, pd=54.9%, 89% HDI=[-0.07410505  0.09094652]


In [13]:
# %% Contrasting HC vs. PTSD
diff_hc_ptsd = slope_hipp_HC - slope_hipp_VPTSD
pd_diff = float((diff_hc_ptsd < 0).mean())  # HC more negative than PTSD
hdi_diff = az.hdi(diff_hc_ptsd.values.flatten(), hdi_prob=0.89)
print(f"HC - VPTSD hipp: mean={float(diff_hc_ptsd.mean()):.3f}, sd = {float(diff_hc_ptsd.std()):.3f}, pd={pd_diff*100:.1f}%, 89% HDI={hdi_diff}")
# %% Contrasting HC vs. VCC
diff_hc_vcc = slope_hipp_HC - slope_hipp_VCC
pd_diff = float((diff_hc_vcc < 0).mean())  # HC more negative than VCC
hdi_diff = az.hdi(diff_hc_vcc.values.flatten(), hdi_prob=0.89)
print(f"HC - VCC hipp: mean={float(diff_hc_vcc.mean()):.3f}, sd = {float(diff_hc_vcc.std()):.3f}, pd={pd_diff*100:.1f}%, 89% HDI={hdi_diff}")

# %% compare contrasts of slopes between HC and PTSD/VCC for vmpfc
diff_vmpfc_ptsd = slope_vmpfc_HC - slope_vmpfc_VPTSD
pd_diff = float((diff_vmpfc_ptsd < 0).mean())  # HC more negative than PTSD
hdi_diff = az.hdi(diff_vmpfc_ptsd.values.flatten(), hdi_prob=0.89)
print(f"HC - VPTSD vmpfc: mean={float(diff_vmpfc_ptsd.mean()):.3f}, sd = {float(diff_vmpfc_ptsd.std()):.3f}, pd={pd_diff*100:.1f}%, 89% HDI={hdi_diff}")
diff_vmpfc_vcc = slope_vmpfc_HC - slope_vmpfc_VCC
pd_diff = float((diff_vmpfc_vcc < 0).mean())  # HC more negative than VCC
hdi_diff = az.hdi(diff_vmpfc_vcc.values.flatten(), hdi_prob=0.89)
print(f"HC - VCC vmpfc: mean={float(diff_vmpfc_vcc.mean()):.3f}, sd = {float(diff_vmpfc_vcc.std()):.3f}, pd={pd_diff*100:.1f}%, 89% HDI={hdi_diff}")

HC - VPTSD hipp: mean=0.041, sd = 0.053, pd=21.6%, 89% HDI=[-0.04438374  0.1235535 ]
HC - VCC hipp: mean=0.036, sd = 0.053, pd=24.3%, 89% HDI=[-0.04812847  0.11977371]
HC - VPTSD vmpfc: mean=-0.079, sd = 0.044, pd=96.4%, 89% HDI=[-0.14544403 -0.0058437 ]
HC - VCC vmpfc: mean=0.038, sd = 0.043, pd=18.6%, 89% HDI=[-0.03736684  0.10093519]
